In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# การปรับแต่ง Transpilation ด้วย SABRE

import TutorialFeedback from '@site/src/components/TutorialFeedback';







*ประมาณเวลาการใช้งาน: 1 นาทีบนโปรเซสเซอร์ Heron r2 (หมายเหตุ: นี่เป็นเพียงการประมาณเท่านั้น เวลาจริงอาจแตกต่างกันได้)*
## ผลการเรียนรู้
หลังจากผ่านบทเรียนนี้ คุณจะเข้าใจ:
- วิธีการกำหนดค่าพารามิเตอร์ SABRE (`layout_trials`, `swap_trials`, `max_iterations`) เพื่อปรับปรุงคุณภาพ transpilation
- การแลกเปลี่ยนระหว่าง runtime ของ transpilation กับคุณภาพวงจร (depth และจำนวน gate)
- วิธีปรับแต่ง routing heuristic ของ SABRE (`basic`, `decay`, `lookahead`) และเปรียบเทียบประสิทธิภาพบนฮาร์ดแวร์

## ข้อกำหนดเบื้องต้น
แนะนำให้คุณคุ้นเคยกับหัวข้อต่อไปนี้ก่อนเรียนบทเรียนนี้:
- [Transpile circuits](/guides/transpile): ภาพรวมของ transpilation ใน Qiskit
- [Transpiler stages](/guides/transpiler-stages): ขั้นตอน layout และ routing
- [Configure preset pass managers](/guides/transpile-with-pass-managers): การปรับแต่ง optimization level

## พื้นหลัง
Transpilation แปลง quantum circuit ให้อยู่ในรูปแบบที่เข้ากันได้กับฮาร์ดแวร์ควอนตัมที่กำหนด สองขั้นตอนสำคัญคือการเลือก **qubit layout** (การแมป logical qubit ไปยัง physical qubit) และ **gate routing** (การแทรก SWAP gate เพื่อให้ multi-qubit gate รองรับข้อจำกัดการเชื่อมต่อของอุปกรณ์)

**SABRE** (*SWAP-Based Bidirectional heuristic search algorithm*) ปรับแต่งทั้ง layout และ routing มีประสิทธิภาพเป็นพิเศษสำหรับวงจรขนาดใหญ่ (100+ qubit) บนอุปกรณ์ที่มี coupling map ซับซ้อน เช่น IBM&reg; Heron processor SABRE ลดจำนวน SWAP gate และลด circuit depth ซึ่งช่วยปรับปรุง execution fidelity การปรับปรุงล่าสุดในอัลกอริทึม [LightSABRE](https://arxiv.org/abs/2409.08368) ช่วยลด runtime และจำนวน gate ลงไปอีก

ในบทเรียนนี้ คุณจะกำหนดค่า `SabreLayout` ด้วยพารามิเตอร์ต่างๆ เพื่อปรับแต่ง GHZ circuit ขนาดเล็กและสังเกต impact ต่อ execution fidelity จากนั้นเปรียบเทียบ routing heuristic ของ SABRE ที่ scale ใหญ่บนฮาร์ดแวร์จริง
## ข้อกำหนด
ก่อนเริ่มบทเรียนนี้ ต้องมีสิ่งต่อไปนี้ติดตั้งอยู่:
- Qiskit SDK v2.0 หรือใหม่กว่า พร้อมรองรับ [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 หรือใหม่กว่า (`pip install qiskit-ibm-runtime`)
- Qiskit Aer (`pip install qiskit-aer`)
## การตั้งค่า

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.transpiler.passes import (
    SabreLayout,
    SabreSwap,
    BarrierBeforeFinalMeasurements,
    StarPreRouting,
)
from qiskit.transpiler.passes.layout.vf2_layout import VF2LayoutStopReason
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.passmanager.flow_controllers import ConditionalController
import matplotlib.pyplot as plt
import numpy as np
import time

seed = 42

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token="<YOUR_API_TOKEN>",  # Replace with your actual API token
    instance="<YOUR_INSTANCE_NAME>",  # Replace with your instance name if needed
)
backend = service.least_busy(operational=True, simulator=False)


print(f"Using backend: {backend.name}")

Using backend: ibm_kingston


## Small-scale simulator example

In this section, a **noisy simulator** based on the real backend's noise model is used to demonstrate how different `SabreLayout` configurations affect both transpilation quality and execution fidelity. Using `qiskit_aer` with a noise model derived from actual hardware calibration data allows you to test the transpilation without consuming hardware credits.

### Step 1: Map classical inputs to a quantum problem

We construct a **star-topology GHZ circuit** with 15 qubits. The first qubit is the hub, with CNOT gates connecting it directly to every other qubit. This topology creates a challenging layout problem because it does not map trivially to the device's coupling map.

We also define `ZZ` operators to measure entanglement correlations $\langle Z_0 Z_i \rangle$ across qubit pairs.

![ghz_star_topology.png](../docs/images/tutorials/transpilation-optimizations-with-sabre/ghz_star_topology.avif)

<Admonition type="note" title="When you know the circuit structure">
SABRE is a general-purpose algorithm and makes no assumptions about circuit structure. For this star-topology GHZ circuit, an optimal routing is actually known: the [`StarPreRouting`](/docs/api/qiskit/qiskit.transpiler.passes.StarPreRouting) pass detects star sub-circuits and rewrites them into a linear chain that maps directly onto any backend with a long enough linear path. This tutorial focuses on SABRE because it works for arbitrary circuits, but if you know your circuit has a clear special structure, applying a specialized pass like `StarPreRouting` before routing can outperform any heuristic search.
</Admonition>

In [2]:
num_qubits_sim = 15

# Create star-topology GHZ circuit
qc_sim = QuantumCircuit(num_qubits_sim)
qc_sim.h(0)
for i in range(1, num_qubits_sim):
    qc_sim.cx(0, i)
qc_sim.measure_all()

# ZZ operators: Z on qubit 0 and qubit i, identity elsewhere
operator_strings_sim = [
    "Z" + "I" * i + "Z" + "I" * (num_qubits_sim - 2 - i)
    for i in range(num_qubits_sim - 1)
]
operators_sim = [SparsePauliOp(op) for op in operator_strings_sim]

## ตัวอย่างขนาดเล็กด้วย simulator
ในส่วนนี้ ใช้ **noisy simulator** ที่อ้างอิง noise model จาก backend จริง เพื่อแสดงให้เห็นว่า configuration ของ `SabreLayout` ต่างๆ ส่งผลต่อคุณภาพ transpilation และ execution fidelity อย่างไร การใช้ `qiskit_aer` กับ noise model ที่ได้จากข้อมูลการสอบเทียบฮาร์ดแวร์จริงช่วยให้คุณทดสอบ transpilation โดยไม่ต้องใช้ hardware credit
### ขั้นตอนที่ 1: แมป classical input ไปยังปัญหาควอนตัม
เราสร้าง **star-topology GHZ circuit** ที่มี 15 qubit โดย qubit แรกทำหน้าที่เป็น hub และมี CNOT gate เชื่อมต่อโดยตรงไปยัง qubit อื่นทุกตัว topology นี้สร้างปัญหา layout ที่ท้าทาย เนื่องจากไม่สามารถแมปตรงๆ ลงบน coupling map ของอุปกรณ์ได้

เรายังกำหนด `ZZ` operator เพื่อวัด entanglement correlation $\langle Z_0 Z_i \rangle$ ข้ามคู่ qubit

![ghz_star_topology.png](../docs/images/tutorials/transpilation-optimizations-with-sabre/ghz_star_topology.avif)

> **Note:** SABRE เป็นอัลกอริทึมทั่วไปและไม่มีข้อสมมติเกี่ยวกับโครงสร้างวงจร สำหรับ star-topology GHZ circuit นี้ มีการ routing ที่เหมาะสมจริงๆ ที่ทราบกัน: pass [`StarPreRouting`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.StarPreRouting) ตรวจจับ star sub-circuit และเขียนใหม่เป็น linear chain ที่แมปได้โดยตรงบน backend ใดๆ ที่มี linear path ยาวพอ บทเรียนนี้เน้นที่ SABRE เพราะมันใช้ได้กับ circuit ทั่วไป แต่ถ้าคุณรู้ว่า circuit มีโครงสร้างพิเศษที่ชัดเจน การใช้ pass เฉพาะทางอย่าง `StarPreRouting` ก่อน routing สามารถทำได้ดีกว่า heuristic search ใดๆ

In [3]:
# Build the default pass manager (no modifications yet)
pm_1 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)

# Visualize the layout stage to see where SabreLayout sits
pm_1.layout.draw()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/b40fe1e0-41cd-4e8b-acb9-801872d35f1f-0.avif" alt="Output of the previous code cell" />

### ขั้นตอนที่ 2: ปรับแต่งปัญหาสำหรับการ execute บนฮาร์ดแวร์ควอนตัม
preset pass manager เริ่มต้น `optimization_level=3` ใช้ `SabreLayout` อยู่แล้ว แต่ด้วยค่าเริ่มต้นที่อนุรักษ์นิยม เพื่อสำรวจผลของการตั้งค่าที่รุนแรงกว่า pass นั้นจะถูกแทนที่ด้วย `SabreLayout` แบบกำหนดเองที่ตั้งค่าสำหรับการค้นหาเชิงรุกมากขึ้น ในขณะที่ทุก pass อื่นใน layout stage ยังคงเดิม นอกจากนี้ยังเพิ่ม pass manager ที่สี่เพื่อเปรียบเทียบ: `pm_star` ที่ใช้ `SabreLayout` เริ่มต้น แต่เพิ่ม [`StarPreRouting`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.StarPreRouting) ใน init stage `StarPreRouting` เป็น pass ที่ *รู้โครงสร้าง* ซึ่งตรวจจับ star sub-circuit และเขียนใหม่เป็น linear chain ก่อน routing

workflow คือ:
1. **ตรวจสอบ** pass manager เริ่มต้นเพื่อดูว่า `SabreLayout` อยู่ที่ไหนใน `layout` stage
2. **แทนที่** pass นั้นด้วย `SabreLayout` instance แบบกำหนดเองโดยใช้ `PassManager.replace(index, passes=...)` และสร้าง `pm_star` variant ด้วย `pm.init += StarPreRouting()`
3. **รัน** pass manager ทั้งสี่และเปรียบเทียบ metric

configuration ทั้งสี่คือ:

| Config | คำอธิบาย |
|--------|-------------|
| `pm_1` (default) | Level-3 preset เริ่มต้น (`SabreLayout` กับ `max_iterations=4`, `layout_trials=20`, `swap_trials=20`) |
| `pm_2` | `SabreLayout` แบบกำหนดเอง (`max_iterations=4`, `layout_trials=200`, `swap_trials=200`) |
| `pm_3` | `SabreLayout` แบบกำหนดเอง (`max_iterations=8`, `layout_trials=200`, `swap_trials=200`) |
| `pm_star` | Preset เริ่มต้นพร้อม `StarPreRouting` เพิ่มใน init stage |

**พารามิเตอร์สำคัญของ SABRE:**
- **`layout_trials` / `swap_trials`**: ควบคุมจำนวน candidate layout และ routing solution ที่ SABRE สำรวจ การเพิ่มจำนวน trial ทำให้ SABRE สุ่มตัวอย่างจากพื้นที่ค้นหาที่กว้างขึ้น เพิ่มโอกาสในการหาวิธีแก้ที่ดีกว่า
- **`max_iterations`**: ควบคุมจำนวน forward-backward routing refinement cycle ที่ SABRE ทำบน candidate แต่ละตัว SABRE ปรับปรุง layout ซ้ำๆ โดยเรียนรู้จาก routing feedback ดังนั้น iteration มากขึ้นหมายถึงการปรับปรุงที่ดีกว่า

ทั้งสองมีต้นทุนที่ transpilation ใช้เวลานานขึ้น แต่วงจรที่ได้จะสั้นกว่าและใช้ gate น้อยกว่า ซึ่งช่วยลด decoherence และ gate error บนฮาร์ดแวร์จริงโดยตรง

**ขั้นตอนที่ 2a: ตรวจสอบ pass manager เริ่มต้น** `StagedPassManager` ประกอบด้วย stage ต่างๆ (`init`, `layout`, `routing`, `translation`, `optimization`, `scheduling`) แต่ละ stage เองก็เป็น `PassManager` การเรียก `.draw()` บน stage จะ render pass เป็นกราฟเพื่อให้เห็นว่า `SabreLayout` อยู่ที่ไหน

In [4]:
cmap = backend.coupling_map

# Custom SabreLayout passes with more aggressive search
sl_2 = SabreLayout(
    coupling_map=cmap,
    seed=seed,
    max_iterations=4,
    layout_trials=200,
    swap_trials=200,
)
sl_3 = SabreLayout(
    coupling_map=cmap,
    seed=seed,
    max_iterations=8,
    layout_trials=200,
    swap_trials=200,
)


# Same condition the preset uses: only run SabreLayout when VF2Layout did not
# find a perfect mapping. This preserves any perfect layout VF2 produced at [1].
def _vf2_match_not_found(property_set):
    if property_set["layout"] is None:
        return True
    return (
        property_set["VF2Layout_stop_reason"] is not None
        and property_set["VF2Layout_stop_reason"]
        is not VF2LayoutStopReason.SOLUTION_FOUND
    )


def wrap_sabre(sabre_pass):
    """Re-wrap a SabreLayout in the original ConditionalController + barrier."""
    return ConditionalController(
        [
            BarrierBeforeFinalMeasurements(
                "qiskit.transpiler.internal.routing.protection.barrier"
            ),
            sabre_pass,
        ],
        condition=_vf2_match_not_found,
    )


# Build two fresh pass managers and swap in the wrapped custom SabreLayout at index 2
pm_2 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_3 = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_2.layout.replace(index=2, passes=wrap_sabre(sl_2))
pm_3.layout.replace(index=2, passes=wrap_sabre(sl_3))

# Build pm_star: default preset with StarPreRouting added to the init stage
pm_star = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=seed
)
pm_star.init += StarPreRouting()

# Visualize pm_3 after replacement (pm_2 has the same structure, only max_iterations differs)
pm_3.layout.draw()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/79075a21-8f36-4fd9-9d0d-bd0e97395b60-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/b40fe1e0-41cd-4e8b-acb9-801872d35f1f-0.avif)

ในไดอะแกรมด้านบน pass `SabreLayout` ที่เราต้องการปรับแต่งอยู่ใน `ConditionalController` ที่ตำแหน่ง **[2]** ของ layout stage controller นี้ทำสองสิ่ง:

- ตรวจสอบ `SabreLayout` ให้รันเฉพาะเมื่อ `VF2Layout` ที่ [1] ไม่พบ perfect mapping (มิฉะนั้น perfect VF2 layout จะถูกเก็บไว้)
- นำหน้า `SabreLayout` ด้วย pass `BarrierBeforeFinalMeasurements` ที่ป้องกันไม่ให้ measurement ถูกจัดลำดับใหม่ระหว่าง routing ภายในของ SabreLayout

ถ้าเพียงแค่ `replace(index=2, passes=sl_2)` พฤติกรรมทั้งสองจะหายไป เพื่อรักษาไว้ เราห่อ `SabreLayout` แบบกำหนดเองของเราใน `ConditionalController` เดิม (พร้อม condition และ protective barrier เดิม) ก่อนสลับเข้าไป

**ขั้นตอนที่ 2b: สร้าง `SabreLayout` pass แบบกำหนดเองและแทนที่ค่าเริ่มต้น**

In [5]:
results_sim = {}
for name, pm in [
    ("pm_1 (4,20,20)", pm_1),
    ("pm_2 (4,200,200)", pm_2),
    ("pm_3 (8,200,200)", pm_3),
    ("pm_star (default + StarPreRouting)", pm_star),
]:
    t0 = time.time()
    tqc = pm.run(qc_sim)
    elapsed = time.time() - t0
    depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
    size = tqc.size()
    ops_mapped = [op.apply_layout(tqc.layout) for op in operators_sim]
    results_sim[name] = {
        "tqc": tqc,
        "ops": ops_mapped,
        "depth": depth,
        "size": size,
        "time": elapsed,
    }
    print(f"{name}: 2Q Depth {depth}, Size {size}, Time {elapsed:.2f}s")

# Print improvement relative to default (pm_1)
baseline = results_sim["pm_1 (4,20,20)"]
print("\nImprovement vs. default (pm_1):")
for name in [
    "pm_2 (4,200,200)",
    "pm_3 (8,200,200)",
    "pm_star (default + StarPreRouting)",
]:
    r = results_sim[name]
    depth_pct = (baseline["depth"] - r["depth"]) / baseline["depth"] * 100
    size_pct = (baseline["size"] - r["size"]) / baseline["size"] * 100
    print(f"  {name}: 2Q depth {depth_pct:+.1f}%, size {size_pct:+.1f}%")

pm_1 (4,20,20): 2Q Depth 38, Size 183, Time 0.01s
pm_2 (4,200,200): 2Q Depth 36, Size 183, Time 0.15s
pm_3 (8,200,200): 2Q Depth 30, Size 158, Time 0.16s
pm_star (default + StarPreRouting): 2Q Depth 26, Size 160, Time 0.01s

Improvement vs. default (pm_1):
  pm_2 (4,200,200): 2Q depth +5.3%, size +0.0%
  pm_3 (8,200,200): 2Q depth +21.1%, size +13.7%
  pm_star (default + StarPreRouting): 2Q depth +31.6%, size +12.6%


![Output of the previous code cell](../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/79075a21-8f36-4fd9-9d0d-bd0e97395b60-0.avif)

ตำแหน่ง **[2]** ตอนนี้เป็น `ConditionalController` อีกครั้ง — มีรูปร่างเหมือนค่าเริ่มต้น แต่ `SabreLayout` ภายในเป็น pass แบบกำหนดเองของเรา (กับ `layout_trials=200`, `swap_trials=200` และ `max_iterations=8` สำหรับ `pm_3`; `pm_2` เหมือนกันยกเว้น `max_iterations=4`) protective barrier และ `_vf2_match_not_found` gating ถูกรักษาไว้ ดังนั้นความแตกต่างเพียงอย่างเดียวระหว่าง `pm_2`/`pm_3` กับ `pm_1` คือ SABRE configuration เอง `pm_star` ใช้ `SabreLayout` เริ่มต้นและเพิ่มเพียง `StarPreRouting` ที่ท้าย init stage

**ขั้นตอนที่ 2c: รัน pass manager แต่ละตัวและเปรียบเทียบ**

In [6]:
pm_names = list(results_sim.keys())
depths = [results_sim[n]["depth"] for n in pm_names]
sizes = [results_sim[n]["size"] for n in pm_names]
times = [results_sim[n]["time"] for n in pm_names]
colors = ["#404080", "#2a9d8f", "#a8d05e", "#e29bdd"]
x = np.arange(len(pm_names))

fig, axs = plt.subplots(1, 3, figsize=(14, 5))

# 2Q Depth
bars = axs[0].bar(x, depths, color=colors)
axs[0].set_ylabel("2Q Depth", fontsize=11)
axs[0].set_title("Two-Qubit Gate Depth", fontsize=13)
axs[0].set_ylim(0, max(depths) * 1.2)
for bar, val in zip(bars, depths):
    axs[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(depths) * 0.02,
        str(val),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
for i in range(1, len(depths)):
    pct = (depths[0] - depths[i]) / depths[0] * 100
    if pct != 0:
        axs[0].text(
            bars[i].get_x() + bars[i].get_width() / 2,
            bars[i].get_height() / 2,
            f"{pct:+.0f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )

# Size
bars = axs[1].bar(x, sizes, color=colors)
axs[1].set_ylabel("Gate Count", fontsize=11)
axs[1].set_title("Circuit Size", fontsize=13)
axs[1].set_ylim(0, max(sizes) * 1.2)
for bar, val in zip(bars, sizes):
    axs[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(sizes) * 0.02,
        str(val),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
for i in range(1, len(sizes)):
    pct = (sizes[0] - sizes[i]) / sizes[0] * 100
    if abs(pct) > 0.1:
        axs[1].text(
            bars[i].get_x() + bars[i].get_width() / 2,
            bars[i].get_height() / 2,
            f"{pct:+.0f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )

# Time
bars = axs[2].bar(x, times, color=colors)
axs[2].set_ylabel("Time (s)", fontsize=11)
axs[2].set_title("Transpilation Time", fontsize=13)
axs[2].set_ylim(0, max(times) * 1.3)
for bar, val in zip(bars, times):
    axs[2].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(times) * 0.03,
        f"{val:.2f}s",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

for ax in axs:
    ax.set_xticks(x)
    ax.set_xticklabels(pm_names, fontsize=8, rotation=15)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle(
    "Transpilation quality vs. configuration",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/bf75c45a-2c3e-4ef6-8336-0b3f69e6e8fb-0.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

We run each transpiled circuit **10 times** using the Aer `EstimatorV2` with a noise model derived from the real backend. Since noisy simulation results vary between runs, averaging over multiple runs gives more reliable fidelity estimates and lets us quantify the statistical uncertainty with error bars.

In [15]:
# Create a noisy estimator from the real backend's noise model
noisy_estimator = AerEstimator.from_backend(backend)

num_runs = 10
# sim_all_runs[name] = list of arrays, one per run
sim_all_runs = {name: [] for name in results_sim}

for run in range(num_runs):
    for name, r in results_sim.items():
        job = noisy_estimator.run([(r["tqc"], r["ops"])])
        evs = list(job.result()[0].data.evs)
        sim_all_runs[name].append(evs)
    print(f"Run {run + 1}/{num_runs} done")

# Compute mean and std across runs for each config
sim_stats = {}
for name in results_sim:
    all_evs = np.array(sim_all_runs[name])  # shape (num_runs, num_operators)
    sim_stats[name] = {
        "mean": np.mean(all_evs, axis=0),
        "std": np.std(all_evs, axis=0),
        "overall_mean": np.mean(all_evs),
        "overall_std": np.std(
            np.mean(all_evs, axis=1)
        ),  # std of per-run averages
    }
    print(
        f"{name}: mean fidelity = {sim_stats[name]['overall_mean']:.4f} +/- {sim_stats[name]['overall_std']:.4f}"
    )

Run 1/10 done
Run 2/10 done
Run 3/10 done
Run 4/10 done
Run 5/10 done
Run 6/10 done
Run 7/10 done
Run 8/10 done
Run 9/10 done
Run 10/10 done
pm_1 (4,20,20): mean fidelity = 0.9510 +/- 0.0094
pm_2 (4,200,200): mean fidelity = 0.9513 +/- 0.0043
pm_3 (8,200,200): mean fidelity = 0.9540 +/- 0.0065
pm_star (default + StarPreRouting): mean fidelity = 0.9547 +/- 0.0072


pass manager ที่ปรับแต่งทั้งสามให้วงจรที่มี 2Q depth ต่ำกว่าค่าเริ่มต้น SABRE configuration แบบรุนแรง (`pm_2` และ `pm_3`) แลกเวลา transpilation ที่นานกว่าเพื่อค้นหาที่กว้างขึ้น ในขณะที่ `pm_star` ใช้ประโยชน์จากโครงสร้าง star ของวงจรและผลิตผลลัพธ์ที่ตื้นกว่าโดยไม่ต้องจ่ายต้นทุน transpilation เพิ่มเติม ผลกำไรที่แน่นอนจะแตกต่างกันในแต่ละรัน แต่แนวโน้มทั่วไปสอดคล้องกัน: SABRE trial และ iteration มากขึ้นทำให้ heuristic ค้นหาพื้นที่กว้างขึ้น และ pass ที่รู้โครงสร้างอย่าง `StarPreRouting` สามารถข้ามการค้นหานั้นได้เลยเมื่อรูปร่างวงจรตรงกัน

แม้ที่ scale เล็กนี้ (15 qubit) ก็มีพื้นที่สำหรับการปรับปรุงเพียงพอที่ทั้งสามวิธีเอาชนะค่าเริ่มต้น ด้วยวงจรขนาดใหญ่ขึ้น (100+ qubit) พื้นที่ค้นหาเติบโตอย่างมาก และประโยชน์ของทั้งการเพิ่ม trial และ pass ที่รู้โครงสร้างจะเด่นชัดยิ่งขึ้นมาก ดังที่ส่วน large-scale จะแสดง

In [16]:
data_sim = list(range(1, len(operators_sim) + 1))
markers = ["o", "s", "^", "*"]
colors_line = ["#404080", "#2a9d8f", "#a8d05e", "#e29bdd"]

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.5, 1]}
)

# Left: correlations vs distance with error bars (mean +/- 1 std)
for (name, stats), marker, color in zip(
    sim_stats.items(), markers, colors_line
):
    ax1.errorbar(
        data_sim,
        stats["mean"],
        yerr=stats["std"],
        marker=marker,
        label=name,
        color=color,
        linewidth=2,
        capsize=3,
        capthick=1,
        elinewidth=1,
    )

ax1.set_xlabel("Distance between qubits $i$", fontsize=11)
ax1.set_ylabel(r"$\langle Z_0 Z_i \rangle$", fontsize=11)
ax1.set_title(
    "Entanglement correlations vs. qubit distance (avg. of 10 runs)",
    fontsize=12,
)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Right: mean correlation bar chart with error bars
names = list(sim_stats.keys())
means = [sim_stats[n]["overall_mean"] for n in names]
stds = [sim_stats[n]["overall_std"] for n in names]
x_bar = np.arange(len(names))
bars = ax2.bar(
    x_bar, means, yerr=stds, color=colors_line, capsize=5, ecolor="gray"
)
ax2.set_ylabel(r"Mean $\langle Z_0 Z_i \rangle$", fontsize=11)
ax2.set_title("Average fidelity", fontsize=13, pad=12)
y_range = max(means) - min(means) if max(means) != min(means) else 0.01
# Top of ylim accounts for the bar height + std error bar + headroom for the value label
y_top = max(m + s for m, s in zip(means, stds)) + y_range * 1.5
ax2.set_ylim(min(means) - y_range * 0.8, y_top)
for bar, val, std in zip(bars, means, stds):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + y_range * 0.15,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )
# Annotate % change vs pm_1
baseline_mean = means[0]
for i in range(1, len(means)):
    pct = (means[i] - baseline_mean) / baseline_mean * 100
    if abs(pct) > 0.01:
        mid_y = (means[i] + ax2.get_ylim()[0]) / 2
        ax2.text(
            bars[i].get_x() + bars[i].get_width() / 2,
            mid_y,
            f"{pct:+.1f}%",
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            fontweight="bold",
        )
ax2.set_xticks(x_bar)
ax2.set_xticklabels(names, fontsize=8, rotation=15)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

fig.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/a6dac5ed-a963-458a-ada1-89c915f036e0-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/bf75c45a-2c3e-4ef6-8336-0b3f69e6e8fb-0.avif)

### ขั้นตอนที่ 3: Execute ด้วย Qiskit primitives
เรารัน circuit ที่ transpiled แต่ละตัว **10 ครั้ง** โดยใช้ Aer `EstimatorV2` กับ noise model ที่ได้จาก backend จริง เนื่องจากผลลัพธ์ noisy simulation แตกต่างกันในแต่ละรัน การเฉลี่ยหลายรันให้การประมาณ fidelity ที่เชื่อถือได้มากขึ้นและช่วยวัด statistical uncertainty ด้วย error bar

In [9]:
# -------------------------Step 1-------------------------

num_qubits = 100

# Create star-topology GHZ circuit
qc = QuantumCircuit(num_qubits)
qc.h(0)
for i in range(1, num_qubits):
    qc.cx(0, i)
qc.measure_all()

# ZZ operators
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (num_qubits - 2 - i)
    for i in range(num_qubits - 1)
]
operators = [SparsePauliOp(op) for op in operator_strings]

In [10]:
# -------------------------Step 2-------------------------

num_seeds = 10
seed_list = [seed + i for i in range(num_seeds)]
swap_trials = 200


# The default routing[1] is a ConditionalController([barrier, routing_pass],
# condition=_swap_condition); we re-wrap so the new routing pass keeps the
# protective barrier and is skipped when routing isn't needed (matches the preset).
def _swap_condition(property_set):
    return not property_set["routing_not_needed"]


def wrap_routing(routing_pass):
    return ConditionalController(
        [
            BarrierBeforeFinalMeasurements(
                "qiskit.transpiler.internal.routing.protection.barrier"
            ),
            routing_pass,
        ],
        condition=_swap_condition,
    )


heuristic_results = {}

# Three SABRE heuristics, swept over seeds
for heuristic in ["basic", "decay", "lookahead"]:
    trials = []
    for s in seed_list:
        sr = SabreSwap(
            coupling_map=cmap, heuristic=heuristic, trials=swap_trials, seed=s
        )
        sl = SabreLayout(coupling_map=cmap, routing_pass=sr, seed=s)
        pm = generate_preset_pass_manager(
            optimization_level=3, backend=backend, seed_transpiler=s
        )
        # Re-wrap each custom pass in its original ConditionalController + barrier
        # (wrap_sabre is defined in the small-scale Step 2 cell above).
        pm.layout.replace(index=2, passes=wrap_sabre(sl))
        pm.routing.replace(index=1, passes=wrap_routing(sr))

        t0 = time.time()
        tqc = pm.run(qc)
        elapsed = time.time() - t0
        depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
        size = tqc.size()
        trials.append(
            {
                "tqc": tqc,
                "depth": depth,
                "size": size,
                "time": elapsed,
                "seed": s,
            }
        )

    heuristic_results[heuristic] = trials

# Default preset + StarPreRouting in init, also swept over seeds for a fair comparison
star_trials = []
for s in seed_list:
    pm_star_hw = generate_preset_pass_manager(
        optimization_level=3, backend=backend, seed_transpiler=s
    )
    pm_star_hw.init += StarPreRouting()

    t0 = time.time()
    tqc = pm_star_hw.run(qc)
    elapsed = time.time() - t0
    depth = tqc.depth(lambda x: x.operation.num_qubits == 2)
    size = tqc.size()
    star_trials.append(
        {
            "tqc": tqc,
            "depth": depth,
            "size": size,
            "time": elapsed,
            "seed": s,
        }
    )
heuristic_results["StarPreRouting"] = star_trials

# Print summary for each entry
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    trials = heuristic_results[label]
    depths = [t["depth"] for t in trials]
    sizes = [t["size"] for t in trials]
    best = min(trials, key=lambda t: t["depth"])
    print(f"{label}:")
    print(
        f"  2Q depth: min: {min(depths)}, mean: {np.mean(depths):.1f}, std: {np.std(depths):.1f}"
    )
    print(
        f"  size    : min: {min(sizes)}, mean: {np.mean(sizes):.1f}, std: {np.std(sizes):.1f}"
    )
    print(
        f"  best seed: {best['seed']} (2Q depth={best['depth']}, size={best['size']})"
    )

basic:
  2Q depth: min: 524, mean: 570.5, std: 39.9
  size    : min: 3819, mean: 4227.1, std: 360.6
  best seed: 51 (2Q depth=524, size=3852)
decay:
  2Q depth: min: 387, mean: 436.4, std: 41.7
  size    : min: 2687, mean: 3183.1, std: 459.3
  best seed: 45 (2Q depth=387, size=2786)
lookahead:
  2Q depth: min: 364, mean: 424.6, std: 36.5
  size    : min: 2335, mean: 3014.6, std: 388.1
  best seed: 51 (2Q depth=364, size=2485)
StarPreRouting:
  2Q depth: min: 196, mean: 196.0, std: 0.0
  size    : min: 1151, mean: 1151.0, std: 0.0
  best seed: 42 (2Q depth=196, size=1151)


In [11]:
hw_colors = {
    "basic": "#ff7f0e",
    "decay": "#d62728",
    "lookahead": "#1f77b4",
    "StarPreRouting": "#2a9d8f",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    trials = heuristic_results[label]
    depths = [t["depth"] for t in trials]
    sizes = [t["size"] for t in trials]
    seeds = [t["seed"] for t in trials]
    color = hw_colors[label]

    ax1.scatter(
        seeds,
        depths,
        label=label,
        color=color,
        alpha=0.8,
        edgecolor="k",
        s=60,
    )
    ax1.axhline(np.mean(depths), color=color, linestyle="--", alpha=0.5)

    ax2.scatter(
        seeds,
        sizes,
        label=label,
        color=color,
        alpha=0.8,
        edgecolor="k",
        s=60,
    )
    ax2.axhline(np.mean(sizes), color=color, linestyle="--", alpha=0.5)

ax1.set_xlabel("Seed", fontsize=11)
ax1.set_ylabel("2Q Depth", fontsize=11)
ax1.set_title("Two-Qubit Gate Depth per Seed", fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

ax2.set_xlabel("Seed", fontsize=11)
ax2.set_ylabel("Gate Count", fontsize=11)
ax2.set_title("Circuit Size per Seed", fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.suptitle(
    "Transpilation variability across seeds: SABRE heuristics vs. StarPreRouting",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

# Summary comparison
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    best = min(heuristic_results[label], key=lambda t: t["depth"])
    print(
        f"{label}: best 2Q depth={best['depth']}, size={best['size']} (seed={best['seed']})"
    )

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/eead9bd2-17e0-4f5b-80bc-eb9b30af052e-0.avif" alt="Output of the previous code cell" />

basic: best 2Q depth=524, size=3852 (seed=51)
decay: best 2Q depth=387, size=2786 (seed=45)
lookahead: best 2Q depth=364, size=2485 (seed=51)
StarPreRouting: best 2Q depth=196, size=1151 (seed=42)


In [12]:
# -------------------------Step 3: Execute on hardware-------------------------

best_circuits = {}
for label in ["basic", "decay", "lookahead", "StarPreRouting"]:
    best_circuits[label] = min(
        heuristic_results[label], key=lambda t: t["depth"]
    )
    b = best_circuits[label]
    print(f"Best {label}: 2Q depth={b['depth']}, size={b['size']}")

options = EstimatorOptions()
options.resilience_level = 2
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"
estimator = Estimator(backend, options=options)

hw_jobs = {}
hw_ops = {}
for label, best in best_circuits.items():
    hw_ops[label] = [op.apply_layout(best["tqc"].layout) for op in operators]
    hw_jobs[label] = estimator.run([(best["tqc"], hw_ops[label])])
    print(f"{label} job: {hw_jobs[label].job_id()}")
estimator.options.environment.job_tags = ["TUT_TOWS"]

hw_results = {}
for label, job in hw_jobs.items():
    hw_results[label] = job.result()[0]
    print(f"{label} job done")

Best basic: 2Q depth=524, size=3852
Best decay: 2Q depth=387, size=2786
Best lookahead: 2Q depth=364, size=2485
Best StarPreRouting: 2Q depth=196, size=1151
basic job: d81q5tnoha1c73bknprg
decay job: d81q5tugbeec73aktopg
lookahead job: d81q5to0bvlc73d1epe0
StarPreRouting job: d81q5u7tjchs73bn82hg
basic job done
decay job done
lookahead job done
StarPreRouting job done


In [13]:
# -------------------------Step 4: Post-process-------------------------

data = list(range(1, len(operators) + 1))
hw_markers = {
    "basic": "D",
    "decay": "o",
    "lookahead": "s",
    "StarPreRouting": "*",
}
hw_labels = ["basic", "decay", "lookahead", "StarPreRouting"]

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.5, 1]}
)

# Left: correlations vs distance
for label in hw_labels:
    evs = list(hw_results[label].data.evs)
    b = best_circuits[label]
    ax1.plot(
        data,
        evs,
        marker=hw_markers[label],
        color=hw_colors[label],
        linewidth=2,
        label=f"{label} (2Q depth={b['depth']}, size={b['size']})",
        markersize=5 if label == "StarPreRouting" else 4,
    )

ax1.set_xlabel("Distance between qubits $i$", fontsize=11)
ax1.set_ylabel(r"$\langle Z_0 Z_i \rangle$", fontsize=11)
ax1.set_title(
    "Entanglement correlations vs. qubit distance (hardware)", fontsize=12
)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Right: mean fidelity bar chart
hw_means = [np.mean(list(hw_results[label].data.evs)) for label in hw_labels]
hw_bar_colors = [hw_colors[label] for label in hw_labels]
x_bar = np.arange(len(hw_labels))
bars = ax2.bar(x_bar, hw_means, color=hw_bar_colors)
ax2.set_ylabel(r"Mean $\langle Z_0 Z_i \rangle$", fontsize=11)
ax2.set_title("Average fidelity", fontsize=13)
y_range = (
    max(hw_means) - min(hw_means) if max(hw_means) != min(hw_means) else 0.01
)
ax2.set_ylim(min(hw_means) - y_range * 0.2, max(hw_means) + y_range * 0.15)
for bar, val in zip(bars, hw_means):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_range * 0.05,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
ax2.set_xticks(x_bar)
ax2.set_xticklabels(hw_labels, fontsize=9, rotation=15)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

fig.tight_layout()
plt.show()

print("\nMean fidelity:")
for label, m in zip(hw_labels, hw_means):
    print(f"  {label}: {m:.4f}")

<Image src="../docs/images/tutorials/transpilation-optimizations-with-sabre/extracted-outputs/0280b0b9-6320-43e5-8396-f82f9e718319-0.avif" alt="Output of the previous code cell" />


Mean fidelity:
  basic: 0.0344
  decay: 0.1298
  lookahead: 0.1857
  StarPreRouting: 0.3295


### Analysis

The scatter plots show significant variability across seeds for all three SABRE heuristics, which underscores the importance of running multiple layout trials rather than relying on a single transpilation. The `StarPreRouting` line is essentially flat across seeds because the rewrite from a star into a linear chain is deterministic given the structure; the downstream SABRE routing then has very little freedom on a linear chain, so the seed has almost no effect on the final depth or size.

From the transpilation results, both the `decay` and `lookahead` heuristics consistently outperform `basic` by a wide margin. The `basic` heuristic, while fast, uses a simple greedy strategy that often leads to substantially deeper circuits. For this star-topology GHZ circuit, `lookahead` tends to produce the lowest 2Q depth and gate count among the SABRE heuristics, since its forward-looking cost function is well suited to circuits with long-range connectivity patterns. `StarPreRouting`, however, dwarfs all three by a substantial margin: by rewriting the star into a linear chain before routing, it short-circuits the search problem entirely and delivers a circuit that the rest of the transpiler can map onto a linear path with minimal additional SWAPs.

That advantage carries straight over to hardware fidelity. Lower 2Q depth and gate count do not always translate one-for-one to higher fidelity (the specific physical qubits a layout uses and their calibration at run time also matter), but when the depth gap is as large as the one between SABRE and `StarPreRouting` here, the structure-aware approach wins decisively because the circuit accumulates far less decoherence and far fewer two-qubit error events. The fidelity bar chart shows `StarPreRouting` substantially ahead of even the best SABRE heuristic, while `basic` sits well below the rest because its much deeper circuits accumulate the most error.

**Key takeaways:**
- Among SABRE heuristics, `decay` and `lookahead` are substantially better than `basic` for non-trivial circuits. Prefer one of the two for production workloads.
- The best SABRE heuristic depends on your circuit and hardware. Testing multiple heuristics with multiple seeds is the most reliable strategy.
- If you want to explore even more layouts, increase `swap_trials` (and `layout_trials` when you are not pinning a custom routing pass) rather than fanning the work out to remote nodes. The SABRE passes already parallelize trials across local threads, and the per-trial work is small enough that distribution overhead typically dominates any speedup.
- When the circuit has a known special structure, applying a structure-aware pass like `StarPreRouting` before SABRE can deliver an order-of-magnitude improvement that no amount of SABRE tuning will match. This is not a replacement for SABRE: `StarPreRouting` only helps when the circuit actually contains star sub-circuits and the backend has a long enough linear path. It is worth checking the pass library for matches whenever you know your circuit's shape.

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [`SabreLayout` API reference](/docs/api/qiskit/qiskit.transpiler.passes.SabreLayout): full parameter documentation
- [SABRE paper](https://arxiv.org/abs/1809.02573): the original SABRE algorithm for layout and routing
- [LightSABRE paper](https://arxiv.org/abs/2409.08368): the algorithmic improvements that power Qiskit's current SABRE implementation
- [Write a custom transpiler pass](/docs/guides/custom-transpiler-pass): build your own transpilation logic
- [Transpiler plugins](/docs/guides/transpiler-plugins): extend Qiskit's transpilation pipeline with third-party passes
- [DAG representation](/docs/guides/DAG-representation): understand the directed acyclic graph used internally by the transpiler
</Admonition>

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_d9YWUSQIAvU9HXE)